# Turkish Daily Dialogues: Kaggle GPU pilot

This runs the project's existing small character-level trainer on the **2,480-record capped train-only derivative** of Turkish Daily Dialogues. It does not use Aya, OASST, Qwen, or upstream validation/test data. The input rows remain marked `draft`: licensing and dataset selection do not mean every generated dialogue has been independently reviewed. The explicit `--allow-draft-data` flag is required and recorded in the run metadata.

## Before running
1. Open this notebook in Kaggle from the public GitHub repository `dozagoat47-star/NextgenAI-Clean-Kaggle-Pilot`.
2. In Notebook settings, enable **Internet** (the setup cell clones the code and train data from GitHub).
3. Select a GPU accelerator (for example T4 or P100 if available), then choose **Run All**. The setup cell stops if CUDA is unavailable, avoiding an accidental long CPU run.

GPU changes training speed, not data quality or model capability. This is a small, synthetic and AI-assisted dataset plus a low-capacity character-level model; treat the result as a pipeline experiment, not a capable Turkish chat model. Each new run needs a fresh output directory. Test evaluation is intentionally disabled.

In [ ]:
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import subprocess
import sys

WORK_ROOT = Path('/kaggle/working') / (
    'nextgenai_clean_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
)
EXPECTED_DATA = Path('data/raw/turkish_daily_dialogues_sft_001/train.jsonl')
EXPECTED_DATA_MANIFEST = Path('data/raw/turkish_daily_dialogues_sft_001/manifest.json')
REPOSITORY_URL = 'https://github.com/dozagoat47-star/NextgenAI-Clean-Kaggle-Pilot.git'

if WORK_ROOT.exists():
    raise FileExistsError(f'Refusing to reuse existing work directory: {WORK_ROOT}')
subprocess.run(
    ['git', 'clone', '--depth', '1', REPOSITORY_URL, str(WORK_ROOT)],
    check=True,
)

bundle = json.loads((WORK_ROOT / 'kaggle_manifest.json').read_text(encoding='utf-8'))
if bundle.get('review_status') != 'draft':
    raise ValueError('Expected draft-status pilot data')
for relative_name, expected_hash in bundle.get('files', {}).items():
    relative_path = Path(relative_name)
    if relative_path.is_absolute() or '..' in relative_path.parts:
        raise ValueError(f'Unsafe manifest path: {relative_name}')
    content = (WORK_ROOT / relative_path).read_bytes()
    canonical_content = content.replace(b'\r\n', b'\n')
    actual_hash = hashlib.sha256(canonical_content).hexdigest()
    if actual_hash != expected_hash:
        raise ValueError(f'GitHub file checksum mismatch: {relative_name}')
data_path = WORK_ROOT / EXPECTED_DATA
data_manifest_path = WORK_ROOT / EXPECTED_DATA_MANIFEST
data_manifest = json.loads(data_manifest_path.read_text(encoding='utf-8'))
if hashlib.sha256(data_path.read_bytes()).hexdigest() != data_manifest.get('training_data_sha256'):
    raise ValueError('Prepared training data does not match its source manifest')
if data_manifest.get('source_selection', {}).get('source_split') != 'train only':
    raise ValueError('Training data manifest is not train-only')

sys.path.insert(0, str(WORK_ROOT))
import torch
from clean_model.data import load_jsonl, split_by_group

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU not detected. Enable a Kaggle GPU accelerator and restart the session.')
records = load_jsonl(data_path, require_approved=False)
if not records or any(record['review']['status'] != 'draft' for record in records):
    raise ValueError('Expected a non-empty, explicitly draft-only dataset')
splits = split_by_group(records, seed=7, require_approved=False)
print('GitHub source:', REPOSITORY_URL)
print('Dataset:', bundle.get('dataset'), '| license:', bundle.get('license'))
print('Records:', len(records), '| groups:', len({r['group_id'] for r in records}))
print('Split record counts:', {name: len(rows) for name, rows in splits.items()})
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda, '| GPU:', torch.cuda.get_device_name(0))
print('Training split only; held-out test split will not be evaluated.')

In [ ]:
# Adjust these pilot settings if needed; keep the seed fixed for comparison.
SEED = 7
EPOCHS = 3
PATIENCE = 2
BATCH_SIZE = 16
D_MODEL = 32
NUM_HEADS = 4
NUM_LAYERS = 1
MAX_SEQ_LEN = 512

RUN_NAME = 'turkish_daily_dialogues_sft_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT_DIR = Path('/kaggle/working') / RUN_NAME
if OUTPUT_DIR.exists():
    raise FileExistsError(f'Refusing to overwrite run: {OUTPUT_DIR}')
command = [
    sys.executable, '-m', 'clean_model.train',
    '--data', str(data_path),
    '--output', str(OUTPUT_DIR),
    '--seed', str(SEED),
    '--epochs', str(EPOCHS),
    '--patience', str(PATIENCE),
    '--batch-size', str(BATCH_SIZE),
    '--d-model', str(D_MODEL),
    '--num-heads', str(NUM_HEADS),
    '--num-layers', str(NUM_LAYERS),
    '--max-seq-len', str(MAX_SEQ_LEN),
    '--allow-draft-data',
]
print('Running:', ' '.join(command))
print('Output directory:', OUTPUT_DIR)

In [ ]:
import subprocess

subprocess.run(command, cwd=WORK_ROOT, check=True)

In [ ]:
metadata_path = OUTPUT_DIR / 'training_metadata.json'
checkpoint_path = OUTPUT_DIR / 'pilot_model.pt'
metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
summary = {
    'run_directory': str(OUTPUT_DIR),
    'checkpoint': str(checkpoint_path),
    'draft_records_included': metadata['draft_records_included'],
    'review_status_counts': metadata['review_status_counts'],
    'split_counts': metadata['split_counts'],
    'split_group_ids': metadata['split_group_ids'],
    'device': metadata['device'],
    'epochs_completed': metadata['training']['epochs_completed'],
    'best_epoch': metadata['best_epoch'],
    'best_validation_response_loss': metadata['best_validation_response_loss'],
    'heldout_test_evaluated': metadata['test_split_evaluated'],
    'checkpoint_sha256': metadata['checkpoint_sha256'],
    'interpretation': metadata['interpretation'],
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

The output folder and checkpoint are under `/kaggle/working`; use Kaggle's **Save Version** or download the run files before the session ends. Do not use the test split to choose hyperparameters. Review generated answers manually and keep this baseline separate from any future verified/approved training corpus.